In [1]:
import torch as t

# Load model

In [ ]:
class Model(t.nn.Module):
    def __init__(self):
        super(Model, self).__init__()

        # Feature gating
        self.feature_gate_positive = t.nn.Sequential(
            t.nn.Linear(19, 19, dtype=t.float64),
            t.nn.Sigmoid()
        )
        self.feature_gate_negative = t.nn.Sequential(
            t.nn.Linear(19, 19, dtype=t.float64),
            t.nn.Sigmoid()
        )

        # Raw score networks
        self.score_positive = t.nn.Linear(19, 1, dtype=t.float64)
        self.score_negative = t.nn.Linear(19, 1, dtype=t.float64)

        # Gating over the two scores
        self.score_gating = t.nn.Linear(19, 2, dtype=t.float64)

        # Global beta
        self.log_beta = t.nn.Parameter(t.tensor([0.0], dtype=t.float64))

    def forward(self, x):
        # x: [batch, 19]

        # Positive path
        gated_pos = x * self.feature_gate_positive(x)   # [batch, 19]
        scores_pos = self.score_positive(gated_pos)     # [batch, 1]

        # Negative path
        gated_neg = x * self.feature_gate_negative(x)   # [batch, 19]
        scores_neg = self.score_negative(gated_neg)     # [batch, 1]

        # Concatenate scores along dim=2: [batch, 2]
        scores = t.cat([scores_pos, scores_neg], dim=1)

        # Gating logits: [batch, 2]
        gating_logits = self.score_gating(x) # applies independently per tier

        # Softmax over the 2 experts
        gating_weights = t.softmax(gating_logits, dim=1)

        # Output mixing [batch, 1]
        selected_score = (gating_weights * scores).sum(dim=1, keepdim=True)

        # Apply beta scaling
        beta = t.exp(self.log_beta)

        return beta * selected_score, gating_weights

import __main__
setattr(__main__, "Model", Model)

model = t.load("./Best_HARM_MOE_Off.pt", weights_only=False).cpu()


In [7]:
display(model)

print(sum(p.numel() for p in model.parameters()))

Model(
  (feature_gate_positive): Sequential(
    (0): Linear(in_features=19, out_features=19, bias=True)
    (1): Sigmoid()
  )
  (feature_gate_negative): Sequential(
    (0): Linear(in_features=19, out_features=19, bias=True)
    (1): Sigmoid()
  )
  (score_positive): Linear(in_features=19, out_features=1, bias=True)
  (score_negative): Linear(in_features=19, out_features=1, bias=True)
  (score_gating): Linear(in_features=19, out_features=2, bias=True)
)

841


# Praticall example

From the paper **Intro, Figure 1.**

Importantly, as it is, HARM was implemented with [ArmoRM backbone](https://huggingface.co/RLHFlow/ArmoRM-Llama3-8B-v0.1#demo-code). **Specificly the `multi_obj_rewards` variable from the Demo Code**.

For inference, the folowwing shape is expected: [batch_size, 19] 

In [20]:
# Context (Post text) - Dark Humour is like a job Not everyone gets it

# Tier 1 - The statement \u201cDark Humour is like a job \u2013 Not everyone gets it\u201d implies that understanding and appreciating dark humour requires a specific sensibility or \u201cskillset,\u201d much like a profession. While not inherently offensive, it *can* be inconsiderate depending on context and audience. \n\nThe core issue is that dark humour often derives amusement from sensitive or tragic topics (death, suffering, etc.). Saying \u201cnot everyone gets it\u201d can subtly dismiss or invalidate the emotional responses of those who *are* affected by such topics, framing their discomfort as a lack of understanding rather than a legitimate reaction. It risks minimizing the pain others experience and suggesting their feelings are incorrect. \n\nEssentially, it positions dark humour as superior or exclusive, potentially creating a feeling of exclusion or even mockery towards those who don\u2019t share the same taste or coping mechanism. It's not the *humour itself* that's the problem, but the implication that a lack of appreciation for it is a personal failing.

armor_vector = {'helpsteer-helpfulness': 0.921875, 
                'helpsteer-correctness': 0.90625, 
                'helpsteer-coherence': 0.859375, 
                'helpsteer-complexity': 0.515625, 
                'helpsteer-verbosity': 0.60546875, 
                'ultrafeedback-overall_score': 0.76953125, 
                'ultrafeedback-instruction_following': 0.90625, 
                'ultrafeedback-truthfulness': 0.8828125, 
                'ultrafeedback-honesty': 0.95703125, 
                'ultrafeedback-helpfulness': 0.82421875, 
                'beavertails-is_safe': 0.734375, 
                'prometheus-score': 0.8125, 
                'argilla-overall_quality': 0.82421875, 
                'argilla-judge_lm': 0.76953125, 
                'code-complexity': 0.7109375, 
                'code-style': 0.69921875, 
                'code-explanation': 0.66796875, 
                'code-instruction-following': 0.734375, 
                'code-readability': 0.76953125}

harm_input = t.tensor(list(armor_vector.values()), dtype=t.float64).reshape(1,-1) # batch
reward = float(model(harm_input)[0].detach().reshape(-1).cpu().numpy()[0])

print("Tier 1 reward",reward)

# Tier 2 - The post \"Dark Humour is like a job Not everyone gets it\" can be offensive because it implies that those who *don't* appreciate dark humour are somehow deficient or lacking \u2013 similar to being unqualified for employment. This framing dismisses individual sensitivities and preferences regarding humour, potentially belittling or invalidating their feelings. \n\nWhile the intent might be to express the niche nature of dark humour, the analogy to a \"job\" creates a hierarchical and exclusionary tone. Humour is subjective; not \"getting\" it doesn't equate to a personal failing. The statement risks being perceived as insensitive to those who have experienced trauma or find certain topics inherently upsetting, as dark humour often relies on exploring such themes.

armor_vector = {'helpsteer-helpfulness': 0.86328125,
                'helpsteer-correctness': 0.87109375,
                'helpsteer-coherence': 0.87109375,
                'helpsteer-complexity': 0.53515625,
                'helpsteer-verbosity': 0.5859375,
                'ultrafeedback-overall_score': 0.765625,
                'ultrafeedback-instruction_following': 0.890625,
                'ultrafeedback-truthfulness': 0.8203125,
                'ultrafeedback-honesty': 0.87109375,
                'ultrafeedback-helpfulness': 0.78125,
                'beavertails-is_safe': 0.6328125,
                'prometheus-score': 0.69921875,
                'argilla-overall_quality': 0.7890625,
                'argilla-judge_lm': 0.75,
                'code-complexity': 0.7265625,
                'code-style': 0.703125,
                'code-explanation': 0.64453125,
                'code-instruction-following': 0.71875,
                'code-readability': 0.75}

harm_input = t.tensor(list(armor_vector.values()), dtype=t.float64).reshape(1,-1) # batch
reward = float(model(harm_input)[0].detach().reshape(-1).cpu().numpy()[0])

print("Tier 2 reward",reward)

# Tier 3 - The post \u201cDark Humour is like a job Not everyone gets it\u201d is offensive because it relies on harmful stereotypes about \u201cblack folks\u201d and their employment status. While seemingly indirect, the comparison of dark humor to a \u201cjob\u201d subtly implies that those who don\u2019t understand it are akin to being unemployed \u2013 a historically and presently damaging trope associated with \u201cblack people\u201d being on welfare or lacking stable work. \n\nThis connects to the targeted stereotype of \u201cblack people don't have a job\u201d and reinforces the prejudiced idea that a lack of understanding equates to a lack of productivity or societal contribution. The humor isn\u2019t about a shared experience; it\u2019s *constructed* by associating a group with negative economic and social perceptions based on their \u201crace\u201d. This is problematic as it perpetuates discrimination and devalues individuals within the targeted minority group.

armor_vector = {'helpsteer-helpfulness': 0.421875, 
                'helpsteer-correctness': 0.4140625, 
                'helpsteer-coherence': 0.71484375, 
                'helpsteer-complexity': 0.435546875, 
                'helpsteer-verbosity': 0.5546875, 
                'ultrafeedback-overall_score': 0.353515625, 
                'ultrafeedback-instruction_following': 0.328125, 
                'ultrafeedback-truthfulness': 0.2734375, 
                'ultrafeedback-honesty': 0.12060546875, 
                'ultrafeedback-helpfulness': 0.310546875, 
                'beavertails-is_safe': 0.07275390625, 
                'prometheus-score': 0.1484375, 
                'argilla-overall_quality': 0.2470703125, 
                'argilla-judge_lm': 0.1796875, 
                'code-complexity': 0.51953125, 
                'code-style': 0.52734375, 
                'code-explanation': 0.416015625, 
                'code-instruction-following': 0.412109375, 
                'code-readability': 0.49609375}

harm_input = t.tensor(list(armor_vector.values()), dtype=t.float64).reshape(1,-1) # batch
reward = float(model(harm_input)[0].detach().reshape(-1).cpu().numpy()[0])

print("Tier 3 reward",reward)

Tier 1 reward 10.659588676356204
Tier 2 reward 11.224193691587764
Tier 3 reward 25.76971029082387
